In [ ]:
!ls

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/Shareddrives/plagiarism_2023

In [ ]:
!pip install evaluate

In [ ]:
!pip install parascore==1.0.5

In [ ]:
!git clone https://github.com/yuh-zha/AlignScore.git

In [ ]:
%cd AlignScore

In [ ]:
!pip install .

In [ ]:
!pip install torchaudio

In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
import nltk
nltk.download('punkt')

In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install datasets

In [ ]:
!pip install bert_score
!pip install mauve-text

## paraphrase evaluation

In [ ]:
from alignscore import AlignScore

scorer = AlignScore(model='roberta-large', batch_size=64,device='cuda', ckpt_path='/content/drive/Shareddrives/plagiarism_2023/Alignscore_model/AlignScore-large.ckpt', evaluation_mode='bin_sp')

In [ ]:

import pandas as pd
%cd /content/drive/Shareddrives/plagiarism_2023/annotation
df = pd.read_json('ref_tldr_paraphrase_gpt3_remaining.json')
df = df.drop_duplicates(subset=['source_doc']).reset_index(drop=True)
df.dropna(subset=['susp_doc'], inplace=True)
print(len(df))

In [ ]:
import torch
from evaluate import load
from transformers import BertTokenizer, BertModel
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline


def get_cola(susp, src):
  cola_model = AutoModelForSequenceClassification.from_pretrained('textattack/roberta-base-CoLA').to("cuda")
  cola_tokenizer = AutoTokenizer.from_pretrained('textattack/roberta-base-CoLA')
  classifier = pipeline('text-classification', model=cola_model, tokenizer=cola_tokenizer, device="cuda")
  tokenizer_kwargs = {'padding':True,'truncation':True}
  src_results = classifier(src, **tokenizer_kwargs)
  susp_results = classifier(susp, **tokenizer_kwargs)

  src_results = [i['label'] for i in src_results]
  susp_results = [i['label'] for i in susp_results]

  return list(zip(susp_results, src_results))


def get_alignscore(claim, doc):
    score = scorer.score(contexts=doc, claims=claim)
    return score


bertscore = load("bertscore")
#mauve = load('mauve')

def get_bertscore(susp, src):
    results = bertscore.compute(predictions=susp, references=src, lang="en", batch_size=32, device="cuda", model_type="microsoft/deberta-xlarge-mnli")
    return results['f1']


def get_mauve(src, susp):
  mauve_results = mauve.compute(predictions=susp, references=src, device_id=0)
  print(mauve_results)
  return mauve_results.mauve

def get_perplexity(susp, src):
  perplexity = load("perplexity", module_type="metric")
  src_truncated = []
  susp_truncated = []
  for i in src:
    src_truncated.append(" ".join(i.split(" ")[:700]))
  for i in susp:
    susp_truncated.append(" ".join(i.split(" ")[:700]))

  results_susp = perplexity.compute(predictions=susp_truncated, model_id='gpt2')
  results_src = perplexity.compute(predictions=src_truncated, model_id='gpt2')

  return list(zip(results_susp['perplexities'], results_src['perplexities']))


from parascore import ParaScorer
parascorer = ParaScorer(lang="en", model_type = 'roberta-large', device="cuda")


def get_parascore(susp, src):
    cands = susp
    sources = src
    refs = src
    score = parascorer.base_score(cands, sources, refs, batch_size=64)
    return score


alignscores = []
bertscores = []
perplexities = []
parascores = []


for i in range(64, len(df), 64):
  end = i
  start = end-64
  print(start, end)
  alignscores.extend(get_alignscore(df['susp_doc'][start:end].tolist(), df['source_doc'][start:end].tolist()))
  bertscores.extend(get_bertscore(df['susp_doc'][start:end].tolist(), df['source_doc'][start:end].tolist()))
  perplexities.extend(get_perplexity(df['susp_doc'][start:end].tolist(), df['source_doc'][start:end].tolist()))
  parascores.extend(get_parascore(df['susp_doc'][start:end].tolist(), df['source_doc'][start:end].tolist()))

  #print(len(alignscores))
  #print(len(bertscores))
  #print(len(perplexities))


  df_new = pd.DataFrame({
    'alignscore': alignscores,
    'bertscore': bertscores,
    'perplexity': perplexities
    'parascore': parascores,
  })

  # Save the DataFrame to a CSV file
  df_new.to_csv("ref_tldr_paraphrase_gpt3_evaluated.csv", index=False)


alignscores.extend(get_alignscore(df['susp_doc'][(len(df)//64)*64:len(df)].tolist(), df['source_doc'][(len(df)//64)*64:len(df)].tolist()))
bertscores.extend(get_bertscore(df['susp_doc'][(len(df)//64)*64:len(df)].tolist(), df['source_doc'][(len(df)//64)*64:len(df)].tolist()))
perplexities.extend(get_perplexity(df['susp_doc'][70*64:len(df)].tolist(), df['source_doc'][70*64:len(df)].tolist()))
parascores.extend(get_parascore(df['susp_doc'][(len(df)//64)*64:len(df)].tolist(), df['source_doc'][(len(df)//64)*64:len(df)].tolist()))

df_new = pd.DataFrame({
    'alignscore': alignscores,
    'bertscore': bertscores,
    'perplexity': perplexities,
    'parascores': parascores
})

# Save the DataFrame to a CSV file
df_new.to_csv("ref_tldr_paraphrase_gpt3_evaluated.csv", index=False)

## summary evaluation

In [ ]:
!pip install datasets unidecode bert_score

In [ ]:
!pip install blanc

In [ ]:
%cd /content/drive/Shareddrives/plagiarism_2023/BARTScore
!pip install -r requirements.txt

In [ ]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("mteb/summeval")
df = pd.DataFrame(dataset['test'])
df = df.explode(['machine_summaries', 'relevance', 'coherence', 'fluency', 'consistency']).reset_index(drop=True)
print(len(df))

In [ ]:
from blanc import BlancHelp, BlancTune
from bart_score import BARTScorer

from evaluate import load
# Step 1: Import the required libraries
from transformers import BertTokenizer, BertModel
import torch
import numpy as np


from alignscore import AlignScore

scorer = AlignScore(model='roberta-large', batch_size=64,device='cuda', ckpt_path='/content/drive/Shareddrives/plagiarism_2023/Alignscore_model/AlignScore-large.ckpt', evaluation_mode='bin_sp')

def get_blancscore(summaries, documents):
    blanc_help = BlancHelp(device='cuda',model_name="bert-large-cased-whole-word-masking",  inference_batch_size=64)
    return blanc_help.eval_pairs(documents, summaries)


from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

def get_cola(src, susp):
  cola_model = AutoModelForSequenceClassification.from_pretrained('textattack/roberta-base-CoLA').to("cuda")
  cola_tokenizer = AutoTokenizer.from_pretrained('textattack/roberta-base-CoLA')
  classifier = pipeline('text-classification', model=cola_model, tokenizer=cola_tokenizer, device="cuda")
  tokenizer_kwargs = {'padding':True,'truncation':True}
  src_results = classifier(src, **tokenizer_kwargs)
  susp_results = classifier(susp, **tokenizer_kwargs)

  src_results = [i['label'] for i in src_results]
  susp_results = [i['label'] for i in susp_results]

  return list(zip(susp_results, src_results))


def get_alignscore(claim, doc):
    score = scorer.score(contexts=doc, claims=claim)
    return score



bart_scorer = BARTScorer(device='cuda:0', checkpoint='facebook/bart-large-cnn')

def get_bartscore(src, susp):
  return bart_scorer.score(srcs=src, tgts=susp, batch_size=16)


def get_perplexity(susp, src):
  perplexity = load("perplexity", module_type="metric")
  src_truncated = []
  susp_truncated = []
  for i in src:
    src_truncated.append(" ".join(i.split(" ")[:700]))
  for i in susp:
    susp_truncated.append(" ".join(i.split(" ")[:700]))

  results_susp = perplexity.compute(predictions=susp_truncated, model_id='gpt2')
  results_src = perplexity.compute(predictions=src_truncated, model_id='gpt2')

  return list(zip(results_susp['perplexities'], results_src['perplexities']))

alignscores = []
blancscores = []
perplexities = []
cola = []
bartscores = []

for i in range(64, len(df), 64):
  end = i
  start = end-64
  print(start, end)
  alignscores.extend(get_alignscore(df['machine_summaries'][start:end].tolist(), df['text'][start:end].tolist()))
  blancscores.extend(get_blancscore(df['machine_summaries'][start:end].tolist(), df['text'][start:end].tolist()))
  bartscores.extend(get_bartscore(df['text'][start:end].tolist(), df['machine_summaries'][start:end].tolist()))
  perplexities.extend(get_perplexity(df['machine_summaries'][start:end].tolist(), df['text'][start:end].tolist()))
  cola.extend(get_cola(df['susp_doc'][start:end].tolist(), df['source_doc'][start:end].tolist()))

  #print(len(alignscores))
  #print(len(bertscores))
  #print(len(perplexities))


  df_new = pd.DataFrame({
    'alignscore': alignscores,
    'blanc': blancscores,
    'bart': bartscores,
    'perplexity': perplexities,
    'cola': cola
  })

  # Save the DataFrame to a CSV file
  df_new.to_csv("summeval_results.csv", index=False)



alignscores.extend(get_alignscore(df['machine_summaries'][(len(df)//64)*64:len(df)].tolist(), df['text'][(len(df)//64)*64:len(df)].tolist()))
blancscores.extend(get_blancscore(df['machine_summaries'][(len(df)//64)*64:len(df)].tolist(), df['text'][(len(df)//64)*64:len(df)].tolist()))
bartscores.extend(get_bartscore(df['text'][(len(df)//64)*64:len(df)].tolist(), df['machine_summaries'][(len(df)//64)*64:len(df)].tolist()))
perplexities.extend(get_perplexity(df['machine_summaries'][(len(df)//64)*64:len(df)].tolist(), df['text'][(len(df)//64)*64:len(df)].tolist()))
cola.extend(get_cola(df['susp_doc'][(len(df)//64)*64:len(df)].tolist(), df['source_doc'][(len(df)//64)*64:len(df)].tolist()))

df_new = pd.DataFrame({
    'alignscore': alignscores,
    'blanc': blancscores,
     'bart': bartscores,
    'perplexity': perplexities,
    #'cola': cola
})

# Save the DataFrame to a CSV file
df_new.to_csv("summeval_results.csv", index=False)